# Libraries

In [1]:
install.packages('KMsurv')
library(KMsurv)
library(survival)
library(ggplot2)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



# Import Data

In [2]:
df <- read.csv('https://raw.githubusercontent.com/farhanage/dataset-for-study/main/heart_failure_clinical_records_dataset.csv')
head(df)

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>
1,75,0,582,0,20,1,265000,1.9,130,1,0,4,1
2,55,0,7861,0,38,0,263358,1.1,136,1,0,6,1
3,65,0,146,0,20,0,162000,1.3,129,1,1,7,1
4,50,1,111,0,20,0,210000,1.9,137,1,0,7,1
5,65,1,160,1,20,0,327000,2.7,116,0,0,8,1
6,90,1,47,0,40,1,204000,2.1,132,1,1,8,1


We will only use `age` as our numerical variable ($x_1$) and `anaemia` as our categorical variable ($x_2$)

In [3]:
df <- df[c('age','anaemia','time','DEATH_EVENT')]
head(df)

,age,anaemia,time,DEATH_EVENT
,<dbl>,<int>,<int>,<int>
1,75,0,4,1
2,55,0,6,1
3,65,0,7,1
4,50,1,7,1
5,65,1,8,1
6,90,1,8,1


We will observe the survival time for each objects to determine the size of the subset that contains a time with ties and a time with no ties

In [4]:
Surv(df$time, df$DEATH_EVENT)

  [1]   4    6    7    7    8    8   10   10   10   10   10   10   11   11   12+
 [16]  13   14   14   15   15   16+  20   20   22+  23   23   24   26   26   26 
 [31]  27   28   28   29+  29   30   30   30   30+  30   31   32   33   33+  33 
 [46]  35   38   40   41   42   43   43   43   44   45   50   54+  54+  55   59 
 [61]  60   60   60+  61   63+  64   65   65   66   67   68+  71+  72   72+  73 
 [76]  73   74+  74+  74+  74+  75+  76+  77   78+  78   79+  79+  79+  79+  79+
 [91]  80+  80+  82+  82   83+  83+  83+  85+  85+  86+  87+  87+  87+  87+  87+
[106]  88   88+  88+  88+  88+  90   90+  90+  90   91+  91+  94+  94+  94+  95 
[121]  95+  95+  95+  95+  96   97+ 100  104+ 104+ 105+ 106+ 107+ 107+ 107+ 107+
[136] 107+ 107+ 108+ 108+ 108+ 109  109+ 109+ 110+ 111  112+ 112+ 113+ 113  115+
[151] 115  117+ 118+ 119+ 120+ 120+ 120+ 120+ 121+ 121+ 121+ 121+ 123+ 126  129 
[166] 130  134+ 135  140+ 145+ 145+ 146+ 146+ 146+ 146+ 146+ 147+ 147+ 147+ 147+
[181] 148+ 150  154  162  17

The first 20 objects consists times with tied events and times with no tied events. 19 events occurred, while 1 object is censored.

In [5]:
subset <- df[1:20,]
print(subset)

   age anaemia time DEATH_EVENT
1   75       0    4           1
2   55       0    6           1
3   65       0    7           1
4   50       1    7           1
5   65       1    8           1
6   90       1    8           1
7   75       1   10           1
8   60       1   10           1
9   65       0   10           1
10  80       1   10           1
11  75       1   10           1
12  62       0   10           1
13  45       1   11           1
14  50       1   11           1
15  49       1   12           0
16  82       1   13           1
17  87       1   14           1
18  45       0   14           1
19  70       1   15           1
20  48       1   15           1


## Parameter Estimation for Data with Ties

After constructing the partial likelihood functions using Breslow Method and Efron Method, we can estimate $\beta_1$ and $\beta_2$ by maximizing the likelihood function.

### Breslow Method

In [6]:
cox <- coxph(Surv(time, DEATH_EVENT) ~ age + anaemia,
                   data = df,
                   subset = c(1:20),
                   ties='breslow')
summary(cox)

Call:
coxph(formula = Surv(time, DEATH_EVENT) ~ age + anaemia, data = df, 
    subset = c(1:20), ties = "breslow")

  n= 20, number of events= 19 

            coef exp(coef) se(coef)      z Pr(>|z|)  
age      0.02214   1.02239  0.01798  1.231   0.2183  
anaemia -1.01140   0.36371  0.58266 -1.736   0.0826 .
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

        exp(coef) exp(-coef) lower .95 upper .95
age        1.0224     0.9781    0.9870     1.059
anaemia    0.3637     2.7494    0.1161     1.140

Concordance= 0.679  (se = 0.089 )
Likelihood ratio test= 3.35  on 2 df,   p=0.2
Wald test            = 3.23  on 2 df,   p=0.2
Score (logrank) test = 3.4  on 2 df,   p=0.2


We get $\hat{\beta_1} = 0.0224$ and $\hat{\beta_2} = -1.01140$

### Efron Method

In [7]:
cox <- coxph(Surv(time, DEATH_EVENT) ~ age + anaemia,
                   data = df,
                   subset = c(1:20),
                   ties='efron')
summary(cox)

Call:
coxph(formula = Surv(time, DEATH_EVENT) ~ age + anaemia, data = df, 
    subset = c(1:20), ties = "efron")

  n= 20, number of events= 19 

            coef exp(coef) se(coef)      z Pr(>|z|)  
age      0.02556   1.02589  0.01791  1.427   0.1535  
anaemia -1.17062   0.31017  0.59392 -1.971   0.0487 *
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

        exp(coef) exp(-coef) lower .95 upper .95
age        1.0259     0.9748   0.99051    1.0625
anaemia    0.3102     3.2240   0.09684    0.9935

Concordance= 0.679  (se = 0.089 )
Likelihood ratio test= 4.35  on 2 df,   p=0.1
Wald test            = 4.15  on 2 df,   p=0.1
Score (logrank) test = 4.44  on 2 df,   p=0.1


We get $\hat{\beta_1} = 0.02556$ and $\hat{\beta_2} = -1.17062$

### Discrete Method

In [8]:
cox <- coxph(Surv(time, DEATH_EVENT) ~ age + anaemia,
                   data = df,
                   subset = c(1:20),
                   ties='exact')
summary(cox)

Call:
coxph(formula = Surv(time, DEATH_EVENT) ~ age + anaemia, data = df, 
    subset = c(1:20), ties = "exact")

  n= 20, number of events= 19 

            coef exp(coef) se(coef)      z Pr(>|z|)  
age      0.02992   1.03037  0.02128  1.406   0.1596  
anaemia -1.36039   0.25656  0.70256 -1.936   0.0528 .
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

        exp(coef) exp(-coef) lower .95 upper .95
age        1.0304     0.9705   0.98829     1.074
anaemia    0.2566     3.8977   0.06474     1.017

Concordance= 0.679  (se = 0.089 )
Likelihood ratio test= 4.44  on 2 df,   p=0.1
Wald test            = 4.07  on 2 df,   p=0.1
Score (logrank) test = 4.4  on 2 df,   p=0.1


We get $\hat{\beta_1} = 0.02992$ and $\hat{\beta_2} = -1.36039$